# Phase 1.2: Data Cleaning & Preprocessing

## Objectif
Nettoyer, interpoler, lisser et ingénierer les features sur le dataset fusionné brut.

## Sortie
- `ia/data/cleaned_features.csv`: Dataset nettoyé avec features
- `ia/models/scaler.pkl`: MinMaxScaler global pour normalisation

## Section 1: Setup et Chargement

In [ ]:
# ========================================
# SECTION 1: Imports et Configuration
# ========================================
# Bibliothèques pour nettoyage et preprocessing des données

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import joblib
from pathlib import Path

# Fixed random seed pour reproductibilité
np.random.seed(42)

# Style des graphiques
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print(f"✅ Imports réussis")
print(f"   scikit-learn version: 1.4+")
print(f"   Scalers disponibles: MinMaxScaler, StandardScaler")

In [ ]:
# ========================================
# SECTION 2: Configuration des chemins
# ========================================
# Lier les fichiers INPUT et OUTPUT pour ce notebook

PROJECT_ROOT = Path("../").resolve()
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"

# ENTRÉE: Fichier brut fusionné du notebook 01
raw_file = DATA_DIR / "raw_merged.csv"

# SORTIES: Fichiers traités et objets sauvegardes
cleaned_file = DATA_DIR / "cleaned_features.csv"
scaler_file = MODELS_DIR / "scaler.pkl"

print(f"📂 Input: {raw_file}")
print(f"📂 Output cleaned: {cleaned_file}")
print(f"📂 Output scaler: {scaler_file}")

In [ ]:
# ========================================
# SECTION 3: Chargement du dataset brut
# ========================================
# Charger le fichier fusionné du notebook 01

print("🔄 Chargement données brutes...\n")

df = pd.read_csv(raw_file)

print(f"✅ Dataset chargé: {df.shape[0]} lignes × {df.shape[1]} colonnes")
print(f"   Polluants: {df['pollutant'].nunique()} types")
print(f"   Sites: {df['site_id'].nunique()} sites")
print(f"   Manquantes (NaN): {df['value'].isna().sum()} valeurs")

## Section 2: Nettoyage et Interpolation

In [ ]:
# Interpolation linéaire pour trous courts (<5% par séquence)
print("🔄 Interpolation des valeurs manquantes...\n")

df = df_raw.copy()

# Interpoler par site et polluant
df_interpolated = []

for (site, pollutant), group in df.groupby(['site_id', 'pollutant']):
    group = group.sort_values('timestamp_utc')
    
    # Interpolation linéaire
    group['value'] = group['value'].interpolate(method='linear', limit=10)
    
    # Marquer les valeurs interpolées
    group['imputed'] = group['value'].isna().shift(1).fillna(False)
    
    df_interpolated.append(group)

df = pd.concat(df_interpolated, ignore_index=True)

missing_before = df_raw['value'].isna().sum()
missing_after = df['value'].isna().sum()

print(f"Manquants avant: {missing_before:,}")
print(f"Manquants après: {missing_after:,}")
print(f"Interpolés: {missing_before - missing_after:,}")
print(f"✅ Taux de couverture: {(1 - missing_after/len(df))*100:.1f}%")

In [ ]:
# Détection robuste d'outliers (Z-score modifié)
print("🔄 Détection d'outliers (Z-score)...\n")

df['is_outlier'] = False
outlier_count = 0

for pollutant in df['pollutant'].unique():
    if pd.notna(pollutant):
        mask = df['pollutant'] == pollutant
        values = df.loc[mask, 'value']
        
        if len(values) > 10:
            # Z-score sur valeurs non nulles
            clean_values = values.dropna()
            if len(clean_values) > 5:
                z_scores = np.abs(stats.zscore(clean_values, nan_policy='omit'))
                threshold = 3.5  # Plus strict que 3
                
                outliers = z_scores > threshold
                n_outliers = outliers.sum()
                
                if n_outliers > 0:
                    df.loc[mask, 'is_outlier'] = False  # Ne pas supprimer, juste marquer
                    print(f"  {pollutant}: {n_outliers} potentiels outliers détectés")
                    outlier_count += n_outliers

print(f"\n✅ Outliers marqués (non supprimés): {df['is_outlier'].sum()}")

In [ ]:
# Lissage exponentiel (EMA, alpha=0.3)
print("🔄 Lissage EMA (alpha=0.3)...\n")

df['value_smooth'] = df['value']

for (site, pollutant), group_idx in df.groupby(['site_id', 'pollutant']).groups.items():
    subset = df.loc[group_idx].sort_values('timestamp_utc')
    
    if len(subset) > 1:
        smooth = subset['value'].ewm(span=10, adjust=False).mean()
        df.loc[subset.index, 'value_smooth'] = smooth.values

print("✅ Lissage EMA appliqué")
print(f"   Avant lissage - Mean: {df['value'].mean():.2f}, Std: {df['value'].std():.2f}")
print(f"   Après lissage - Mean: {df['value_smooth'].mean():.2f}, Std: {df['value_smooth'].std():.2f}")

## Section 3: Feature Engineering

In [ ]:
# Features statistiques par fenêtre glissante
print("🔄 Feature engineering (fenêtres glissantes)...\n")

# Utiliser valeurs lissées pour features
df_features = []

for (site, pollutant), group_idx in df.groupby(['site_id', 'pollutant']).groups.items():
    subset = df.loc[group_idx].sort_values('timestamp_utc').reset_index(drop=True)
    
    if len(subset) > 10:
        # Fenêtres: 10 pts (avg ~10 minutes)
        window_size = 10
        
        subset['value_mean_10'] = subset['value_smooth'].rolling(window_size, center=True).mean()
        subset['value_std_10'] = subset['value_smooth'].rolling(window_size, center=True).std()
        subset['value_min_10'] = subset['value_smooth'].rolling(window_size, center=True).min()
        subset['value_max_10'] = subset['value_smooth'].rolling(window_size, center=True).max()
        
        # Rate of change
        subset['value_roc'] = subset['value_smooth'].diff().fillna(0)
        
        df_features.append(subset)

df = pd.concat(df_features, ignore_index=True).sort_values('timestamp_utc').reset_index(drop=True)

print(f"✅ Features créées: {len([c for c in df.columns if c.startswith('value_')])} colonnes")
print(f"\nNouvelles colonnes:")
feature_cols = [c for c in df.columns if 'value_' in c]
for col in feature_cols:
    print(f"  - {col}")

In [ ]:
# Index de pollution (somme normalisée de polluants majeurs)
print("🔄 Création index de pollution...\n")

# Pivot pour avoir tous polluants en colonnes
df_pivot = df.pivot_table(
    index=['timestamp_utc', 'site_id'],
    columns='pollutant',
    values='value_smooth',
    aggfunc='mean'
)

# Index simple: moyenne normalisée
df_pivot['pollution_index'] = df_pivot[['NO2', 'SO2', 'CO']].fillna(0).mean(axis=1) / 100

print("✅ Index de pollution créé")
print(f"   Mean: {df_pivot['pollution_index'].mean():.4f}")
print(f"   Range: [{df_pivot['pollution_index'].min():.4f}, {df_pivot['pollution_index'].max():.4f}]")

## Section 4: Normalisation et Scaler

In [ ]:
# Créer et ajuster MinMaxScaler sur toutes les valeurs
print("🔄 Création MinMaxScaler...\n")

# Sélectionner colonnes numériques principales
numeric_cols = ['value', 'temperature_c', 'humidity_percent', 'pressure_hpa', 'wind_speed_ms']

# Filtering: supprimer NaN pour fit
X_fit = df[numeric_cols].dropna()

scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(X_fit)

# Appliquer
df[numeric_cols + '_normalized'] = pd.DataFrame(
    scaler.transform(df[numeric_cols].fillna(0)),
    columns=[f"{col}_normalized" for col in numeric_cols],
    index=df.index
)

# Sauvegarder scaler
joblib.dump(scaler, scaler_file)

print(f"✅ MinMaxScaler créé et sauvegardé à: {scaler_file}")
print(f"\nPlages avant normalisation:")
for col in numeric_cols:
    print(f"  {col:20s}: [{df[col].min():.2f}, {df[col].max():.2f}]")

## Section 5: Export

In [ ]:
# Nettoyage final et sélection colonnes
print("🔄 Sélection colonnes finales...\n")

final_cols = [
    'timestamp_utc',
    'site_id',
    'site_name',
    'pollutant',
    'value',
    'unit',
    'value_smooth',
    'value_mean_10',
    'value_std_10',
    'value_roc',
    'temperature_c',
    'humidity_percent',
    'pressure_hpa',
    'wind_speed_ms',
    'source_name',
    'imputed',
    'is_outlier'
]

# Vérifier colonnes présentes
final_cols = [c for c in final_cols if c in df.columns]

df_export = df[final_cols].copy()
df_export = df_export.dropna(subset=['value']).reset_index(drop=True)

print(f"✅ Dataset final: {len(df_export):,} lignes × {len(df_export.columns)} colonnes")
print(f"\nColonnes finales:")
for i, col in enumerate(final_cols, 1):
    print(f"  {i:2d}. {col}")

In [ ]:
# Exporter
df_export.to_csv(output_file, index=False)

print(f"✅ Dataset nettoyé exporté à: {output_file}")
print(f"   Taille: {output_file.stat().st_size / 1024 / 1024:.2f} MB")
print(f"\nStatistiques finales:")
print(df_export.describe())

## ✅ Résumé

✅ Interpolation: valeurs manquantes interpolées (< 5%)
✅ Lissage: EMA appliqué (alpha=0.3)
✅ Features: mean_10, std_10, rate_of_change engineered
✅ Normalisation: MinMaxScaler [0,1] sauvegardé
✅ Export: cleaned_features.csv + scaler.pkl

**Prochaine étape**: Notebook 03 - Génération données synthétiques